# Zero-Error State Overlaps

Check the overlap at each finite sensor threshold using sparse eigen-components. Reads saved inputs without running allocation solvers.


## Method

For a nonzero phase gap, let $n=N_{ijm}$ and $\theta=|r_{ijm}|/2$. Use three constructions:

1. If $n|r_{ijm}|=\pi$, use $(|A_+\rangle+|A_-\rangle)/\sqrt{2}$.
2. If $n|r_{ijm}|>\pi$ and $n$ is even, use the $|A_+\rangle, |A_-\rangle, |B_0\rangle$ construction.
3. If $n|r_{ijm}|>\pi$ and $n$ is odd, use the $|A_+\rangle, |A_-\rangle, |B_+\rangle, |B_-\rangle$ construction.

The code evaluates the overlap from the nonzero eigen-components of these states. This avoids constructing a full $2^n$ state vector, which is necessary because very small phase gaps can produce very large thresholds.

In [ ]:
from pathlib import Path
import importlib
import sys

from IPython.display import Markdown, display

working_directory = Path.cwd().resolve()
if (working_directory / "problem-1-zero-error").is_dir():
    PROJECT_DIRECTORY = working_directory / "problem-1-zero-error"
elif working_directory.name == "sanity-check":
    PROJECT_DIRECTORY = working_directory.parent
else:
    PROJECT_DIRECTORY = working_directory

SANITY_DIRECTORY = PROJECT_DIRECTORY / "sanity-check"
INPUT_ROOT = PROJECT_DIRECTORY / "inputs"
OUTPUT_ROOT = SANITY_DIRECTORY / "outputs"

if str(SANITY_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SANITY_DIRECTORY))

import overlap_sanity
overlap_sanity = importlib.reload(overlap_sanity)

print("Project directory:", PROJECT_DIRECTORY)
print("Input root:", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)

## Run settings

Use `RUN_SCOPE = "test"` for a small subset or `"full"` for both datasets. Detailed per-location output is optional.


In [ ]:
DATASETS = ("benchmark_1d", "benchmark_2d")

RUN_SCOPE = "full"
if RUN_SCOPE not in {"test", "full"}:
    raise ValueError('RUN_SCOPE must be "test" or "full"')

CASE_LIMIT = None if RUN_SCOPE == "full" else 10
PROGRESS_EVERY = 250 if RUN_SCOPE == "full" else 1

OVERLAP_TOLERANCE = 1e-12
BOUNDARY_TOLERANCE = 1e-12

WRITE_DETAILED_PAIR_LOCATION_CSV = False

print("Run scope:", RUN_SCOPE)
print("Case limit per dataset:", CASE_LIMIT)
print("Progress interval:", PROGRESS_EVERY, "cases")
print("Overlap tolerance:", OVERLAP_TOLERANCE)
print("Write detailed CSV:", WRITE_DETAILED_PAIR_LOCATION_CSV)

## Compute overlaps

Save summaries, violations, and samples under `outputs/`.


In [ ]:
results = overlap_sanity.run_sanity_check(
    INPUT_ROOT,
    OUTPUT_ROOT,
    datasets=DATASETS,
    case_limit=CASE_LIMIT,
    overlap_tolerance=OVERLAP_TOLERANCE,
    boundary_tolerance=BOUNDARY_TOLERANCE,
    progress_every=PROGRESS_EVERY,
    write_detailed_csv=WRITE_DETAILED_PAIR_LOCATION_CSV,
)

print("Wrote sanity-check outputs to:", results["output_root"])

## Summary

Maximum, mean, and RMS overlap. Violations exceed the configured numerical tolerance.


In [ ]:
def markdown_table(rows, columns):
    header = "| " + " | ".join(label for label, _ in columns) + " |"
    separator = "|" + "|".join("---" for _ in columns) + "|"
    body = []
    for row in rows:
        values = []
        for _, key in columns:
            value = row[key]
            if isinstance(value, float):
                values.append(f"{value:.3e}")
            else:
                values.append(str(value))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([header, separator, *body])

summary_columns = [
    ("Dataset", "dataset"),
    ("Cases", "cases"),
    ("Finite pair-locations", "finite_pair_locations"),
    ("Zero-gap locations", "zero_gap_unusable_locations"),
    ("Max |overlap|", "max_overlap_abs"),
    ("Mean |overlap|", "mean_overlap_abs"),
    ("RMS |overlap|", "rms_overlap_abs"),
    ("Violations", "violation_count"),
]
display(Markdown(markdown_table(results["summary"], summary_columns)))

## Construction counts

Counts by construction case; zero phase gaps have no finite threshold.


In [ ]:
construction_columns = [
    ("Dataset", "dataset"),
    ("Boundary", "boundary_cases"),
    ("Above threshold, even n", "above_threshold_even_n_cases"),
    ("Above threshold, odd n", "above_threshold_odd_n_cases"),
]
display(Markdown(markdown_table(results["summary"], construction_columns)))

## Largest residual

The pair and location with the largest absolute overlap in each dataset.


In [ ]:
worst_columns = [
    ("Dataset", "dataset"),
    ("Case", "worst_case"),
    ("Pair", "worst_pair"),
    ("Location", "worst_location"),
    ("N_ijm", "worst_N_ijm"),
    ("Max |overlap|", "max_overlap_abs"),
]
display(Markdown(markdown_table(results["summary"], worst_columns)))

## Violations

An empty table means all finite overlaps passed the tolerance check.


In [ ]:
if results["violations"]:
    violation_columns = [
        ("Dataset", "dataset"),
        ("Case", "case"),
        ("Pair", "i"),
        ("j", "j"),
        ("Location", "m"),
        ("N_ijm", "N_ijm"),
        ("Construction", "construction_case"),
        ("|overlap|", "overlap_abs"),
    ]
    display(Markdown(markdown_table(results["violations"][:20], violation_columns)))
else:
    display(Markdown("No overlap violations were found."))

## Output files


In [ ]:
for path in sorted(OUTPUT_ROOT.glob("*.csv")):
    print(path)